# 逐层搭建 YOLO26s、视觉 Mamba 和 ViT

本笔记本直接定义分支，组装三种融合检测器，并和训练入口构建的模型逐项比较。默认在 CPU 上运行，无需数据集。请在项目根目录打开，内核选择本项目的 `.venv-cuda`。最后一节导出五组模型的结构。训练时运行仓库根目录的 train.py，完成的实验自动写入 Excel。

## 1. 载入同一份 YOLO26s 初始权重

先将原来的 80 类检测器重建成两类检测器。三组实验共享它的骨干、Neck 和检测头。

In [1]:
from copy import deepcopy
from pathlib import Path
import math
import sys

import torch
from torch import nn
from torch.nn import functional as F
from torchvision.models.vision_transformer import EncoderBlock
sys.path.insert(0, str(Path.cwd() / "training/support"))
from fusion.runtime import ARTIFACTS, PROJECT, bootstrap

if Path(sys.prefix).resolve() != (PROJECT / ".venv-cuda").resolve() and sys.prefix == sys.base_prefix:
    raise RuntimeError("请将 Notebook 切换到本项目的独立训练环境。")
bootstrap()
from fusion.models import build_model, make_detector

torch.set_num_threads(2)
WEIGHTS = ARTIFACTS / "weights/yolo26s.pt"
def detector(name):
    initialized, _ = build_model(name, seed=0, yolo_weights=WEIGHTS)
    return make_detector(initialized.model.yaml, nc=2, weights=initialized.model).cpu().eval()

baseline = detector("baseline")
for i in (10, 13):
    print("原网络第", i, "层", type(baseline.model[i]).__name__)

Overriding model.yaml nc=80 with nc=2


原网络第 10 层 C2PSA
原网络第 13 层 C3k2


## 2. 写出选择性状态扫描

一个位置的状态等于前一个状态乘以衰减系数，再加上当前位置写入的信息。下面的函数把相邻位置两两合并，得到与顺序递推一致的结果。运行示例会逐项检查。

In [2]:
def affine_scan(decay, drive):
    """All states h[t] = decay[t] * h[t-1] + drive[t], with h[-1] = 0."""
    length = drive.size(-1)
    if length == 1:
        return drive
    pairs = length // 2
    left = slice(0, 2 * pairs, 2)
    right = slice(1, 2 * pairs, 2)
    odd = affine_scan(decay[..., right] * decay[..., left],
                      drive[..., right] + decay[..., right] * drive[..., left])
    even = torch.cat((drive[..., :1], drive[..., 2::2] + decay[..., 2::2] * odd[..., :(length - 1) // 2]), -1)
    interleaved = torch.stack((even[..., :pairs], odd), -1).flatten(-2)
    return torch.cat((interleaved, even[..., pairs:]), -1) if length % 2 else interleaved

In [3]:
decay = torch.tensor([0.5, 0.5, 0.2, 0.8, 0.4])
drive = torch.tensor([1., 2., 0., -1., 3.])
states = []
h = torch.tensor(0.)
for a, u in zip(decay, drive):
    h = a * h + u
    states.append(h)
torch.testing.assert_close(affine_scan(decay, drive), torch.stack(states))
print("各位置状态", affine_scan(decay, drive).tolist())

各位置状态 [1.0, 2.5, 0.5, -0.5999999642372131, 2.759999990463257]


## 3. 在二维特征图上实现四向 Mamba 残差分支

读 `forward` 时沿特征的变化顺序看。卷积先保留局部邻域，再把特征依行、依列以及它们的反方向展开。每个位置从输入得到步长及写入、读取权重。四条路径扫描完成后分别还原到原图坐标，合并并投影回原通道。可学习的 `alpha` 控制新分支进入 YOLO 主路的幅度。

In [4]:
class ResidualSS2D(nn.Module):
    def __init__(self, base, channels, width, state_size):
        super().__init__()
        self.base = base
        self.in_proj = nn.Conv2d(channels, width, 1)
        self.local = nn.Conv2d(width, width, 3, padding=1, groups=width)
        self.rank = math.ceil(width / 16)
        self.state_size = state_size
        self.select = nn.Parameter(torch.empty(4, self.rank + 2 * state_size, width))
        self.step = nn.Parameter(torch.empty(4, width, self.rank))
        nn.init.uniform_(self.select, -width ** -0.5, width ** -0.5)
        nn.init.uniform_(self.step, -self.rank ** -0.5, self.rank ** -0.5)
        dt = torch.empty(4, width).uniform_(math.log(.001), math.log(.1)).exp()
        self.step_bias = nn.Parameter(dt.expm1().log())
        self.log_A = nn.Parameter(torch.arange(1, state_size + 1).float().log().repeat(4, width, 1))
        self.skip = nn.Parameter(torch.ones(4, width))
        self.norm = nn.LayerNorm(width)
        self.out_proj = nn.Conv2d(width, channels, 1)
        self.alpha = nn.Parameter(torch.zeros(()))
        self.i, self.f, self.type = base.i, base.f, type(self).__name__
        self.np = sum(p.numel() for p in self.parameters())

    def forward(self, x):
        base = self.base(x)
        z = F.silu(self.local(self.in_proj(base)))
        b, c, h, w = z.shape
        row = z.flatten(2)
        col = z.transpose(2, 3).flatten(2)
        seq = torch.stack((row, col, row.flip(-1), col.flip(-1)), 1)
        select = torch.einsum("bkcl,kpc->bkpl", seq, self.select)
        steps, write, read = select.split((self.rank, self.state_size, self.state_size), dim=2)
        dt = F.softplus(torch.einsum("bkrl,kcr->bkcl", steps, self.step) + self.step_bias[..., None])
        decay = torch.exp(-dt.unsqueeze(3) * self.log_A.exp()[..., None])
        drive = (dt * seq).unsqueeze(3) * write.unsqueeze(2)
        state = affine_scan(decay, drive)
        y = (state * read.unsqueeze(2)).sum(3) + seq * self.skip[..., None]
        rows = (y[:, 0] + y[:, 2].flip(-1)).reshape(b, c, h, w)
        cols = (y[:, 1] + y[:, 3].flip(-1)).reshape(b, c, w, h).transpose(2, 3)
        branch = self.norm(((rows + cols) / 4).permute(0, 2, 3, 1)).permute(0, 3, 1, 2)
        return base + self.alpha * self.out_proj(branch)

## 4. 定义 P5 上的小型 ViT

把 P5 特征的每个空间位置作为一个 token，投影到 256 维，加入二维位置编码，经过 4 头注意力后再还原成特征图。这个分支既可以独立使用，也可以和第 13 层的 Mamba 组合。

In [5]:
class P5ResidualViT(nn.Module):
    def __init__(self, base, dim, heads):
        super().__init__()
        self.base = base
        self.input_projection = nn.Conv2d(512, dim, 1)
        self.encoder = EncoderBlock(heads, dim, dim * 4, dropout=0.0, attention_dropout=0.0)
        self.output_projection = nn.Conv2d(dim, 512, 1)
        self.alpha = nn.Parameter(torch.zeros(()))
        self.dim = dim
        # Ultralytics routes layers by these attributes, including the P5 skip connection.
        self.i, self.f, self.type = base.i, base.f, "P5ResidualViT"
        self.np = sum(p.numel() for p in self.parameters())

    def position(self, h, w, device, dtype):
        freq = 10000.0 ** (-torch.arange(self.dim // 4, device=device).float() / (self.dim // 4))
        y, x = torch.meshgrid(torch.arange(h, device=device), torch.arange(w, device=device), indexing="ij")
        x, y = x.flatten()[:, None] * freq, y.flatten()[:, None] * freq
        return torch.cat((x.sin(), x.cos(), y.sin(), y.cos()), -1).to(dtype).unsqueeze(0)

    def forward(self, x):
        x = self.base(x)
        features = self.input_projection(x)
        n, c, h, w = features.shape
        tokens = features.flatten(2).transpose(1, 2)
        tokens = self.encoder(tokens + self.position(h, w, tokens.device, tokens.dtype))
        branch = self.output_projection(tokens.transpose(1, 2).reshape(n, c, h, w))
        return x + self.alpha * branch

## 5. 亲手更改 YOLO26s 的三种层结构

下面每行赋值都会实际替换检测器中的对应层。包装器先调用原层，再加上残差分支，因此保留了原有特征和跨层连接。

In [6]:
def assemble(name):
    net = deepcopy(baseline)
    if name in ("p5_vit_small", "p4_mamba_p5_vit"):
        net.model[10] = P5ResidualViT(net.model[10], dim=256, heads=4)
    if name == "p5_mamba":
        net.model[10] = ResidualSS2D(net.model[10], channels=512, width=64, state_size=8)
    if name in ("p4_mamba", "p4_mamba_p5_vit"):
        net.model[13] = ResidualSS2D(net.model[13], channels=256, width=64, state_size=8)
    net.variant = name
    net.yaml["fusion_variant"] = name
    return net.eval()

variants = ("p5_vit_small", "p5_mamba", "p4_mamba", "p4_mamba_p5_vit")
for name in variants:
    net = assemble(name)
    print(name, " | 第 10 层", type(net.model[10]).__name__,
          " | 第 13 层", type(net.model[13]).__name__,
          " | 参数量", sum(p.numel() for p in net.parameters()))

p5_vit_small  | 第 10 层 P5ResidualViT  | 第 13 层 C3k2  | 参数量 11002085
p5_mamba  | 第 10 层 ResidualSS2D  | 第 13 层 C3k2  | 参数量 10024997
p4_mamba  | 第 10 层 C2PSA  | 第 13 层 ResidualSS2D  | 参数量 9991973


p4_mamba_p5_vit  | 第 10 层 P5ResidualViT  | 第 13 层 ResidualSS2D  | 参数量 11044646


## 6. 用真实前向计算核对通道、尺度和零门控

特征图打印为 (批次, 通道, 高, 宽)。640×640 的图像应产生 20×20 的 P5 和 40×40 的 P4。四个分支方案初始门控均为零，因此检测输出应与同权重的原版完全相同。

In [7]:
image = torch.rand(1, 3, 640, 640)
with torch.inference_mode():
    reference = baseline(image)[0]
    for name in variants:
        net = assemble(name)
        shapes = {}
        def capture(index):
            def record(module, inputs, output):
                shapes[index] = tuple(output.shape)
            return record
        hooks = [net.model[i].register_forward_hook(capture(i)) for i in (10, 13)]
        try:
            prediction = net(image)[0]
        finally:
            for hook in hooks:
                hook.remove()
        torch.testing.assert_close(prediction, reference, rtol=0, atol=0)
        print(name, "P5", shapes[10], "P4", shapes[13], "零门控输出一致")

p5_vit_small P5 (1, 512, 20, 20) P4 (1, 256, 40, 40) 零门控输出一致
p5_mamba P5 (1, 512, 20, 20) P4 (1, 256, 40, 40) 零门控输出一致


p4_mamba P5 (1, 512, 20, 20) P4 (1, 256, 40, 40) 零门控输出一致


p4_mamba_p5_vit P5 (1, 512, 20, 20) P4 (1, 256, 40, 40) 零门控输出一致


## 7. 打开分支，核对笔记本与训练脚本的实现

在矩形输入上给两个版本载入同一参数，并将两个残差门控临时设为 0.2。断言通过说明分支真正参与计算时，手工组装与正式训练入口的输出一致。这个演示只改内存里的模型。正式训练仍从零门控开始。正式训练使用仓库中可导入的类，权重可在独立 Python 进程重新载入。

In [8]:
rectangle = torch.rand(1, 3, 128, 224)
with torch.inference_mode():
    for name in variants:
        manual, training_model = assemble(name), detector(name)
        manual.load_state_dict(training_model.state_dict(), strict=True)
        for net in (manual, training_model):
            for i in (10, 13):
                if hasattr(net.model[i], "alpha"):
                    net.model[i].alpha.fill_(0.2)
        torch.testing.assert_close(manual(rectangle)[0], training_model(rectangle)[0], rtol=0, atol=0)
        print(name, "矩形输入与训练实现一致")

Overriding model.yaml nc=80 with nc=2


p5_vit_small

 矩形输入与训练实现一致


Overriding model.yaml nc=80 with nc=2


p5_mamba 矩形输入与训练实现一致


Overriding model.yaml nc=80 with nc=2


p4_mamba 矩形输入与训练实现一致


Overriding model.yaml nc=80 with nc=2


p4_mamba_p5_vit 矩形输入与训练实现一致


## 8. 保存五组可训练的模型结构

把上面实际组装的检测器导出到一个 YAML 文件。每组保存完整 YOLO 层连接、分支所在层及超参数。训练脚本读取这些结构，并从同一份 COCO 初始权重开始训练；新增分支根据当前种子初始化。此处生成的文件可以重复使用。

In [9]:
import yaml
from fusion.runtime import MODEL_SET

definitions = {}
for name in ("baseline", *variants):
    net = baseline if name == "baseline" else assemble(name)
    cfg = deepcopy(net.yaml)
    cfg.pop("yaml_file", None)
    cfg.pop("fusion_seed", None)
    cfg.pop("fusion_deterministic", None)
    cfg["nc"] = 80  # COCO 初始权重的类别数，训练器随后按数据集重建两类检测头
    cfg["fusion_variant"] = name
    spec = {}
    if isinstance(net.model[10], P5ResidualViT):
        vit = net.model[10]
        spec.update(dim=vit.dim, heads=vit.encoder.self_attention.num_heads)
    for index in (10, 13):
        branch = net.model[index]
        if isinstance(branch, ResidualSS2D):
            spec.update(scan_layer=index, width=branch.in_proj.out_channels, state_size=branch.state_size)
    cfg["fusion_spec"] = spec
    definitions[name] = cfg

MODEL_SET.parent.mkdir(parents=True, exist_ok=True)
temporary = MODEL_SET.with_suffix(".yaml.tmp")
temporary.write_text(yaml.safe_dump(definitions, sort_keys=False), encoding="utf-8")
temporary.replace(MODEL_SET)
print(MODEL_SET.relative_to(PROJECT), list(definitions))

models\fusion.yaml ['baseline', 'p5_vit_small', 'p5_mamba', 'p4_mamba', 'p4_mamba_p5_vit']


## 9. 在终端启动训练

在仓库根目录的 PowerShell 中运行下面的命令。从 YOLO26s 基线开始，五组模型依次各训练 100 轮。

```powershell
& .\.venv-cuda\Scripts\python.exe train.py all --data training/configs/study-data.yaml
```

只训练 P5 Mamba 时，把 `all` 换成 `fit --model p5_mamba`。完成后打开 [实验结果汇总.xlsx](training/实验结果汇总.xlsx)，查看整体、UGV 和 USV 的验证集 mAP。